In [ ]:
# The BASE_URL of any openai-compatible endpoint
# Task: fill the base_url
BASE_URL = "..."

# Model Tag, find it by navigating to the OVH Model Catalog and filtering "Structured Output"
# https://www.ovhcloud.com/de/public-cloud/ai-endpoints/catalog/
# Task: Find a model that supports structured output
MODEL_TAG = "..."

In [ ]:
# Import all the necessary libraries
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from pydantic import BaseModel, Field
from typing import List
import os

from dotenv import load_dotenv
import json

load_dotenv()  # in Codespaces the keys come from the secrets, locally from .env

# Structured Output

In [ ]:
# Define a simple ... Message
# Task: What message type is needed here?
messages = [
    ...(
        content="""
        Du bist ein hilfreicher Assistent, der bei der Extraktion von Versicherungsdokumenten hilft.
        """
    )
]

In [ ]:
# Define the content to extract from

document = """
Versicherungsnehmer: Max Mustermann
Anschrift: Musterweg 12, 12345 Musterstadt
Versicherungsgegenstand: Einfamilienhaus in Musterstadt, Baujahr 2000, Wohnfläche 150 m²
Deckungsumfang: Feuerschäden, Leitungswasser, Sturm/Hagel, Einbruchdiebstahl, Glasbruch
Selbstbeteiligung: 150 € pro Schadensfall
Laufzeit: 01.01.2026 bis 31.12.2026 (ein Jahr)
Kündigungsfristen: Zum Vertragsende mit einer Frist von drei Monaten; außerordentliche Kündigung bei erhöhtem Risiko möglich
Ausschlüsse: Erdbebenschäden, Krieg, Kernenergie, vorsätzlich herbeigeführte Schäden, Schäden durch grobe Fahrlässigkeit bei Nichtbeachtung von Sicherheitsvorschriften
"""

In [ ]:
# Task: Fill the gaps

class Versicherungsbedingung(...):
    vorname: ... = Field(
        name="Vorname",
        description="Vorname des Versicherungsnehmers"
    )
    nachname: ... = Field(
        name="Nachname",
        description="Nachname des Versicherungsnehmers"
    )
    strasse: str = Field(
        ...="Straße",
        description="Straßenname der Anschrift"
    )
    hausnummer: ... = Field(
        name="Hausnummer",
        description="Hausnummer der Anschrift"
    )
    plz: ... = Field(
        name="Postleitzahl",
        description="Fünfstellige PLZ der Anschrift"
    )
    ort: str = Field(
        name="Ort",
        description="Stadt oder Gemeinde der Anschrift"
    )
    versicherungsgegenstand: str = Field(
        name="Versicherungsgegenstand",
        description="Beschreibung des versicherten Objekts (z.B. Gebäude, Fahrzeug, Inventar)"
    )
    deckungsumfang: ...[str] = Field(
        name="Deckungsumfang",
        description="Liste der versicherten Gefahren (z.B. Feuer, Leitungswasser, Sturm)"
    )
    selbstbeteiligung: str = Field(
        name="Selbstbeteiligung",
        description="Betrag, den der Versicherungsnehmer im Schadensfall selbst trägt"
    )
    laufzeit: str = Field(
        name="Laufzeit",
        description="Versicherungsperiode im Format TT.MM.JJJJ bis TT.MM.JJJJ"
    )
    kuendigungsfristen: ...[str] = Field(
        name="Kündigungsfristen",
        description="Liste der Kündigungsregelungen (ordentlich, außerordentlich etc.)"
    )
    ausschluesse: ...[str] = Field(
        name="Ausschlüsse",
        description="Liste von Risiken oder Schäden, die vom Versicherungsschutz ausgeschlossen sind"
    )

In [ ]:
# Create an LLM Instance with LangChain
# Task: Create an OpenAI compatible llm_instance
# The OVH key is in os.environ["OVH_API_KEY"] (Codespaces secret or .env)
llm_instance = ...

In [ ]:
# Make it Structured Output capable
llm_structured = llm_instance.with_structured_output(Versicherungsbedingung)

In [ ]:
# Invoke the LLM to obtain your response
response = llm_structured.invoke(messages + [HumanMessage(content=document)])

# Check the type of your response
print(type(...))
print(response.model_dump_json(indent=2))

# Manuelle Extraktion (Prompt)

In [ ]:
prompt = """
Extrahiere aus dem folgenden Versicherungsbedingungstext die relevanten Informationen und gib das Ergebnis ausschließlich als JSON‑Objekt zurück. Verwende genau diese Schlüsselnamen und Typen:

vorname (string)

nachname (string)

strasse (string)

hausnummer (string)

plz (string)

ort (string)

versicherungsgegenstand (string)

deckungsumfang (array of strings – jede versicherte Gefahr einzeln)

selbstbeteiligung (string)

laufzeit (string)

kuendigungsfristen (array of strings – jede Kündigungsregelung einzeln)

ausschluesse (array of strings – jeder Ausschluss einzeln)

Antworte ausschließlich mit dem JSON‑Objekt, ohne zusätzlichen Text.

Dokumenteninhalt:

{document}
"""

In [ ]:
response_normal = llm_instance.invoke(messages + [HumanMessage(content=prompt.format(document=document))])

print(type(response_normal))
print(response_normal.content)